# AML fraud detection: baseline GNN

This notebook trains a small edge-classification GNN on the verified graph artifact. It is designed for Google Colab and uses the chronological masks created by the preprocessing pipeline.

## Colab setup

Upload or copy `data/parquet/graph_data.pt` to Google Drive first. The graph is large, so Drive is preferred over browser upload.

In [ ]:
!pip -q install torch-geometric scikit-learn

from google.colab import drive
drive.mount('/content/drive')

# Update this path if your Drive folder is different.
GRAPH_PATH = '/content/drive/MyDrive/AML-Fraud-Detection/data/parquet/graph_data.pt'

In [ ]:
import random
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_fscore_support, precision_recall_curve
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# PyTorch 2.6+ defaults to weights_only=True; graph_data.pt contains a PyG Data object.
data = torch.load(GRAPH_PATH, map_location='cpu', weights_only=False)
print(data)
print('nodes:', data.x.shape[0], 'edges:', data.edge_index.shape[1])

## Model

Two GCN layers build node representations; each directed edge is then classified from its endpoint embeddings plus its stored edge attributes.

**Temporal message passing.** The model never sees future transactions while training:

| Phase | Message-passing graph | Edges scored |
|---|---|---|
| train | train edges | train |
| validate | train + val edges | val |
| test | all edges | test |

Logits are computed only for the edges being scored, which keeps memory well below a full 5M-edge forward. The loss uses `pos_weight` computed from the training mask only.

In [ ]:
class EdgeGCN(torch.nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=64):
        super().__init__()
        self.conv1 = GCNConv(node_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.edge_head = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim * 2 + edge_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, message_edge_index, target_edge_index, target_edge_attr):
        # Embed nodes using only the edges visible in this phase...
        h = F.relu(self.conv1(x, message_edge_index))
        h = F.dropout(h, p=0.2, training=self.training)
        h = self.conv2(h, message_edge_index)
        # ...then score only the target edges.
        src, dst = target_edge_index
        edge_repr = torch.cat([h[src], h[dst], target_edge_attr], dim=-1)
        return self.edge_head(edge_repr).squeeze(-1)

# Chronological masks: train edges precede val edges precede test edges.
graphs = {
    'train': data.edge_index[:, data.train_mask],
    'val': data.edge_index[:, data.train_mask | data.val_mask],
    'test': data.edge_index,
}
targets = {
    split: (data.edge_index[:, mask], data.edge_attr[mask], data.y[mask])
    for split, mask in [('train', data.train_mask), ('val', data.val_mask), ('test', data.test_mask)]
}
x = data.x.to(device)
graphs = {k: v.to(device) for k, v in graphs.items()}
targets = {k: tuple(t.to(device) for t in v) for k, v in targets.items()}
for split in ['train', 'val', 'test']:
    print(f"{split}: message edges={graphs[split].size(1):,} target edges={targets[split][2].numel():,} positives={int(targets[split][2].sum()):,}")

model = EdgeGCN(data.x.size(-1), data.edge_attr.size(-1)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
train_y = targets['train'][2]
train_pos = train_y.sum()
train_neg = train_y.numel() - train_pos
pos_weight = (train_neg / train_pos.clamp_min(1)).float()
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print('training positives:', int(train_pos), 'training negatives:', int(train_neg), 'pos_weight:', float(pos_weight))

## Training

Full-batch training means one optimizer step per epoch, so many epochs are needed. Validation average precision (PR-AUC) drives early stopping and checkpoint selection: at a ~0.1% positive rate it is far more informative than ROC-AUC.

The decision threshold is **not** 0.5. With a large `pos_weight`, scores are shifted upward, so the threshold is chosen to maximize F1 on validation and then applied unchanged to test.

In [ ]:
@torch.no_grad()
def predict(split):
    model.eval()
    edge_index, edge_attr, labels = targets[split]
    scores = model(x, graphs[split], edge_index, edge_attr).sigmoid()
    return labels.long().cpu().numpy(), scores.cpu().numpy()

def best_f1_threshold(labels, scores):
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    best = int(np.argmax(f1[:-1]))  # the final PR point has no threshold
    return float(thresholds[best]), float(f1[best])

def metrics(labels, scores, threshold):
    predictions = (scores >= threshold).astype(np.int64)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', zero_division=0
    )
    return {
        'average_precision': average_precision_score(labels, scores),
        'roc_auc': roc_auc_score(labels, scores),
        'threshold': threshold, 'precision': precision, 'recall': recall, 'f1': f1,
        'flagged': int(predictions.sum()), 'positives': int(labels.sum()),
        'positive_rate': float(labels.mean()),
    }

MAX_EPOCHS = 500
EVAL_EVERY = 5
PATIENCE = 20  # evaluations without val-AP improvement (= 100 epochs)

train_edge_index, train_edge_attr, _ = targets['train']
best_ap, best_epoch, best_state, stale = -1.0, 0, None, 0
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits = model(x, graphs['train'], train_edge_index, train_edge_attr)
    loss = criterion(logits, train_y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
    optimizer.step()
    if epoch % EVAL_EVERY == 0 or epoch == 1:
        val_ap = average_precision_score(*predict('val'))
        improved = val_ap > best_ap
        if improved:
            best_ap, best_epoch, stale = val_ap, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
        print(f"epoch {epoch:03d} loss={loss.item():.4f} val_AP={val_ap:.4f}{'  *best' if improved else ''}")
        if stale >= PATIENCE:
            print(f"early stop: no val-AP improvement for {PATIENCE * EVAL_EVERY} epochs")
            break

model.load_state_dict(best_state)
print(f"restored best model from epoch {best_epoch} (val AP={best_ap:.4f})")

In [ ]:
# Threshold is chosen on validation only, then applied unchanged to test.
val_labels, val_scores = predict('val')
threshold, val_best_f1 = best_f1_threshold(val_labels, val_scores)
print('val metrics: ', metrics(val_labels, val_scores, threshold))
test_labels, test_scores = predict('test')
print('test metrics:', metrics(test_labels, test_scores, threshold))

CHECKPOINT_PATH = '/content/drive/MyDrive/AML-Fraud-Detection/edge_gcn_baseline.pt'
torch.save({
    'model_state_dict': model.state_dict(), 'seed': SEED, 'pos_weight': float(pos_weight),
    'best_epoch': best_epoch, 'val_average_precision': best_ap, 'threshold': threshold,
}, CHECKPOINT_PATH)
print('saved:', CHECKPOINT_PATH)